API KEY

In [3]:
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv(usecwd=True)

if not dotenv_path:
    raise FileNotFoundError("Arquivo .env não encontrado.")

load_dotenv(dotenv_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY não encontrada dentro do arquivo .env.")

os.environ["OPENAI_API_KEY"] = api_key

print("OPENAI_API_KEY carregada com sucesso.")

OPENAI_API_KEY carregada com sucesso.


IMPORTS

In [4]:
import os
import json
from typing import Dict, List
from litellm import completion

GENERATE RESPONSE FUNCTION

In [5]:
def generate_response(messages: List[Dict]) -> str:
    """Call the LLM and return the response content."""
    response = completion(
        model="openai/gpt-4o-mini",
        messages=messages,
        max_tokens=1024
    )
    return response.choices[0].message.content

CREATING TOOLS

In [19]:
# List files
def list_files() -> List[str]:
    """List all files in the current directory."""
    return [
        file for file in os.listdir(".")
        if os.path.isfile(file)
    ]

# Read files
def read_file(file_name: str) -> str:
    """Read the content of a file."""
    with open(file_name, "r", encoding="utf-8") as file:
        return file.read()

EXTRATING THE ACTION BLOCK

In [18]:
def extract_markdown_block(response: str, block_type: str) -> str:
    """Extract content from a markdown block like ```action ... ```."""
    start_marker = f"```{block_type}"
    end_marker = "```"

    start_index = response.find(start_marker)

    if start_index == -1:
        raise ValueError(f"No {block_type} block found.")

    start_index += len(start_marker)
    end_index = response.find(end_marker, start_index)

    if end_index == -1:
        raise ValueError(f"No closing markdown block found for {block_type}.")

    return response[start_index:end_index].strip()

CREATING THE PARSE FUNCTION

In [16]:
def parse_action(response: str) -> Dict:
    """Parse the LLM response into a structured action dictionary."""
    try:
        response = extract_markdown_block(response, "action")
        response_json = json.loads(response)

        if "tool_name" in response_json and "args" in response_json:
            return response_json

        return {
            "tool_name": "error",
            "args": {"message": "Valid JSON, but missing 'tool_name' or 'args'."},
        }

    except json.JSONDecodeError:
        return {"tool_name": "error", "args": {"message": "Invalid JSON response."}}

    except ValueError as error:
        return {"tool_name": "error", "args": {"message": str(error)}}

CREATING INITIAL RULES AND MEMORY

In [24]:
agent_rules = [
    {
        "role": "system",
        "content": """
You are an AI agent that can perform tasks by using available tools.

Available tools:
- list_files() -> List[str]: List all files in the current directory.
- read_file(file_name: str) -> str: Read the content of a file.
- terminate(message: str): End the agent loop and print a summary to the user.

Rules:
- If the user asks about files, list them before reading.
- Every response MUST contain exactly one action.
- Do not answer directly unless you are using the terminate tool.
- Use terminate only when you have completed the task.

Respond only in this format:

```action
{
    "tool_name": "insert_tool_name",
    "args": {}
}
```
    
""",
    }
]

memory = [
    {
        "role": "user",
        "content": "Please check my files and summarize the content of them.",
    }
]

AGENT LOOP

In [25]:
iterations = 0
max_iterations = 10

while iterations < max_iterations:
    prompt = agent_rules + memory

    print("\nAgent thinking...")
    response = generate_response(prompt)
    print(f"\nAgent response:\n{response}")

    action = parse_action(response)
    print(f"\nParsed action:\n{action}")

    if action["tool_name"] == "list_files":
        result = {"result": list_files()}

    elif action["tool_name"] == "read_file":
        result = {"result": read_file(action["args"]["file_name"])}

    elif action["tool_name"] == "error":
        result = {"error": action["args"]["message"]}

    elif action["tool_name"] == "terminate":
        print("\nFinal answer:")
        print(action["args"]["message"])
        break

    else:
        result = {"error": "Unknown action: " + action["tool_name"]}

    print(f"\nAction result:\n{result}")

    memory.extend(
        [
            {"role": "assistant", "content": response},
            {"role": "user", "content": json.dumps(result)},
        ]
    )

    iterations += 1


Agent thinking...

Agent response:
```action
{
    "tool_name": "list_files",
    "args": {}
}
```

Parsed action:
{'tool_name': 'list_files', 'args': {}}

Action result:
{'result': ['TextFile2', 'HandingTextFilesAgent.ipynb', 'TextFile1']}

Agent thinking...

Agent response:
```action
{
    "tool_name": "read_file",
    "args": {"file_name": "TextFile2"}
}
```

Parsed action:
{'tool_name': 'read_file', 'args': {'file_name': 'TextFile2'}}

Action result:
{'result': 'Finish the AI agent course.\nTest the file-reading tool.\nImprove the agent memory loop.'}

Agent thinking...

Agent response:
```action
{
    "tool_name": "read_file",
    "args": {"file_name": "HandingTextFilesAgent.ipynb"}
}
```

Parsed action:
{'tool_name': 'read_file', 'args': {'file_name': 'HandingTextFilesAgent.ipynb'}}

Action result:
{'result': ''}

Agent thinking...

Agent response:
```action
{
    "tool_name": "read_file",
    "args": {"file_name": "TextFile1"}
}
```

Parsed action:
{'tool_name': 'read_file', 'a

In [ ]:
prompt = agent_rules + memory

print("\nAgent thinking...")
response = generate_response(prompt)
print(f"\nAgent response:\n{response}")


Agent thinking...

Agent response:
```action
{
    "tool_name": "list_files",
    "args": {}
}
```


In [ ]:
action = parse_action(response)
print(f"\nParsed action:\n{action}")


Parsed action:
{'tool_name': 'list_files', 'args': {}}


In [22]:
if action["tool_name"] == "list_files":
        result = {"result": list_files()}

elif action["tool_name"] == "read_file":
        result = {"result": read_file(action["args"]["file_name"])}

elif action["tool_name"] == "error":
        result = {"error": action["args"]["message"]}

#elif action["tool_name"] == "terminate":
#        print("\nFinal answer:")
#        print(action["args"]["message"])
#        break

else:
        result = {"error": "Unknown action: " + action["tool_name"]}

print(f"\nAction result:\n{result}")


Action result:
{'result': ['TextFile2', 'HandingTextFilesAgent.ipynb', 'TextFile1']}
